# Project: Compare CNN and ViT (no position enc) against each other!

This notebook will be the primary vehicle for creating, training, evaluating the models.

# 0. Initial Library imports

From the project root, run `bash setup_venv.sh`, then `source .venv/bin/activate`
and `jupyter lab src/project.ipynb`. Select the **Python (coms474)** kernel.

In [2]:
# Standard library: paths, downloads, reproducibility, and timing
import json
import random
import time
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

# Data analysis and visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

# PyTorch models, optimization, image transforms, and data loading
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, models, transforms

# 1. Dataset download & Setup

In [3]:
# Download TinyImageNet once; reuse the extracted data or cached ZIP on reruns.
# Resolve src/data whether Jupyter starts in the project root or in src.
PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "src" / "project.ipynb").is_file()
)
DATA_DIR = PROJECT_ROOT / "src" / "data"
DATASET_DIR = DATA_DIR / "tiny-imagenet-200"
ARCHIVE_PATH = DATA_DIR / "tiny-imagenet-200.zip"
DATASET_URL = "https://cs231n.stanford.edu/tiny-imagenet-200.zip"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def dataset_is_ready():
    # Check image counts as well as directories to detect interrupted extraction.
    required = [DATASET_DIR / "wnids.txt", DATASET_DIR / "words.txt",
                DATASET_DIR / "val" / "val_annotations.txt"]
    if not all(path.is_file() for path in required):
        return False
    wnids = required[0].read_text().splitlines()
    return (
        len(set(wnids)) == 200
        and all(len(list((DATASET_DIR / "train" / wnid / "images").glob("*.JPEG"))) == 500
                for wnid in wnids)
        and len(list((DATASET_DIR / "val" / "images").glob("*.JPEG"))) == 10_000
    )


if not dataset_is_ready():
    if not ARCHIVE_PATH.is_file():
        # Rename only after a successful download so interrupted files aren't reused.
        partial_path = ARCHIVE_PATH.with_suffix(".zip.part")
        print(f"Downloading {DATASET_URL} ...")
        urlretrieve(DATASET_URL, partial_path)
        partial_path.replace(ARCHIVE_PATH)

    print(f"Extracting {ARCHIVE_PATH.name} ...")
    with ZipFile(ARCHIVE_PATH) as archive:
        # Keep every archive member inside the data directory.
        for name in archive.namelist():
            if not (DATA_DIR / name).resolve().is_relative_to(DATA_DIR.resolve()):
                raise ValueError(f"Unsafe archive path: {name}")
        archive.extractall(DATA_DIR)

    if not dataset_is_ready():
        raise RuntimeError("TinyImageNet extraction is incomplete. Check the cached ZIP and rerun.")

print(f"TinyImageNet is ready at {DATASET_DIR}")


Extracting tiny-imagenet-200.zip ...
TinyImageNet is ready at /home/tyler/coms474/src/data/tiny-imagenet-200


In [ ]:
# Shared data pipeline for the CNN and ViT: 100,000 train / 10,000 validation.
SEED = 67
BATCH_SIZE = 128
NUM_WORKERS = 0 # default
IMAGE_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Fixed ImageNet normalization.
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.RandomCrop(IMAGE_SIZE, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


train_dataset = datasets.ImageFolder(DATASET_DIR / "train", transform=train_transform)
class_to_idx = train_dataset.class_to_idx
classes = train_dataset.classes
NUM_CLASSES = len(classes)


class TinyImageNetValidation(Dataset):
    """Read the official flat validation layout without moving or copying images."""

    def __init__(self, root, class_to_idx, transform):
        self.transform = transform
        self.class_to_idx = dict(class_to_idx)
        self.classes = sorted(class_to_idx, key=class_to_idx.get)
        self.samples = []
        for line in (root / "val_annotations.txt").read_text().splitlines():
            filename, wnid, *_ = line.split()
            self.samples.append((root / "images" / filename, class_to_idx[wnid]))
        self.targets = [target for _, target in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
        return self.transform(image), target


val_dataset = TinyImageNetValidation(DATASET_DIR / "val", class_to_idx, eval_transform)
words = dict(line.split("\t", 1) for line in (DATASET_DIR / "words.txt").read_text().splitlines())
class_names = [words[wnid] for wnid in classes]


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_train_loader(seed=SEED):
    return DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
    )


train_loader = make_train_loader()
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
    worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Device: {DEVICE}; classes: {NUM_CLASSES}; batch size: {BATCH_SIZE}")
print(f"Training images: {len(train_dataset):,}; validation images: {len(val_dataset):,}")
print(f"Model input shape: (batch, 3, {IMAGE_SIZE}, {IMAGE_SIZE})")


Device: cuda; classes: 200; batch size: 128
Training images: 100,000; validation images: 10,000
Model input shape: (batch, 3, 64, 64)


# 2. Convolutional Neural Network

In [ ]:
# TODO Define our layered model

In [ ]:
# TODO Train our model (maybe a few different hyperparams to be safe and get best model)

In [ ]:
# TODO Evaluate Our Model (Save our evals to output/cnn/)

In [ ]:
# TODO Save our model (.pt or weights)

# 3. Vision Transformer (no positional encoding)

In [ ]:
# TODO Define our model (num heads, size, etc) [Param count should be similar to CNN]

In [ ]:
# TODO Train our model(s)

In [ ]:
# TODO Evaluate our model(s) (save evals to output/vit/)

In [ ]:
# TODO Save our model (.pt or weights)

# 4. Comparisons

In [ ]:
# TODO Training Time Comparison

In [ ]:
# TODO Evaluation Comparison

In [ ]:
# TODO Bias Comparison

In [ ]:
# TODO Variance Comparison